# 🔄 03 - Metadata-Only Schema Evolution

### 📌 Objective
In this notebook, we demonstrate **Schema Evolution** in Apache Iceberg:
1. **Add a new column** (`category STRING`).
2. **Rename an existing column** (`word_count` ➡️ `word_count_v2`).
3. **Append new records** with the new schema.
4. **Query old and new records together** seamlessly without rewriting any underlying Parquet files!

### 💡 How Iceberg Schema Evolution Works:
- Hive tables use **column position** in Parquet files. If you add or rename columns, old files get corrupted or misread.
- Iceberg tracks columns by **unique Field IDs**. When you rename a column, only the metadata alias changes. The Parquet files remain untouched!

In [ ]:
from pyspark.sql import SparkSession

# Re-use or connect to existing Spark session
spark = SparkSession.builder.getOrCreate()

# Check current table schema
print("--- Current Table Schema ---")
spark.sql("DESCRIBE TABLE icelake.wikipedia.articles").show(truncate=False)

### Step 1: Evolve Schema (Add Column & Rename Column)
These commands execute in **milliseconds** because they only update `metadata.json` in S3.

In [ ]:
# Check if column category already exists, if not add it
try:
    spark.sql("ALTER TABLE icelake.wikipedia.articles ADD COLUMNS (category STRING)")
    print("✅ Added column 'category'")
except Exception as e:
    print("ℹ️ Column 'category' already exists:", str(e)[:100])

# Check if word_count is present to rename
try:
    spark.sql("ALTER TABLE icelake.wikipedia.articles RENAME COLUMN word_count TO word_count_v2")
    print("✅ Renamed 'word_count' to 'word_count_v2'")
except Exception as e:
    print("ℹ️ Column already renamed:", str(e)[:100])

print("\n--- Updated Table Schema ---")
spark.sql("DESCRIBE TABLE icelake.wikipedia.articles").show(truncate=False)

### Step 2: Query Old Data vs New Data
Notice how old records from before the schema evolution have `category = NULL` and their word counts correctly mapped under `word_count_v2`!

In [ ]:
print("Old data (category is NULL):")
spark.sql("""
    SELECT id, title, created_date, word_count_v2, category 
    FROM icelake.wikipedia.articles 
    WHERE category IS NULL 
    LIMIT 5
""").show(truncate=False)

print("New data (category is populated):")
spark.sql("""
    SELECT id, title, created_date, word_count_v2, category 
    FROM icelake.wikipedia.articles 
    WHERE category IS NOT NULL 
    LIMIT 5
""").show(truncate=False)